# Model Evaluation (Old Model)

In this section, we evaluate the performance of the trained YOLO model on the validation dataset.

The goal is to measure how well the model detects cars using key metrics:
- Precision
- Recall
- mAP@50
- mAP@50-95

In [1]:
from ultralytics import YOLO
import json

# Load the trained model
model = YOLO("best.pt")

## Running Evaluation

We evaluate the model using the validation dataset.

### Parameters:
- **imgsz**: Image size used during evaluation
- **conf**: Confidence threshold
- **iou**: IoU threshold for detection matching

In [2]:
metrics = model.val(
    data="data.yaml",
    split="val",
    imgsz=640,
    conf=0.25,
    iou=0.6,
    plots=True
)

Ultralytics 8.4.33 🚀 Python-3.9.6 torch-2.4.1 CPU (Apple M1)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 337.7±56.4 MB/s, size: 75.7 KB)
val: Scanning /Users/slom/Downloads/My First Project.yolov8 2/dataset/labels.cache... 104 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 104/104 4.0Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 1227, len(boxes) = 14646. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 4.1s/it 28.7s5.0ss
                   all        104      14646      0.768      0.628      0.688      0.559
Speed: 2.7ms preprocess, 263.5ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /Users/slom/ru

## Evaluation Results

The following metrics are used:

- **Precision**: How many detected cars are correct
- **Recall**: How many actual cars were detected
- **mAP@50**: Detection performance at IoU = 0.5
- **mAP@50-95**: More strict and comprehensive evaluation

In [3]:
results = {
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map)
}

print("\n=== Evaluation Results ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


=== Evaluation Results ===
precision: 0.7682
recall: 0.6284
mAP50: 0.6877
mAP50_95: 0.5588


## Saving Results

The evaluation results are saved as a JSON file for later use in reporting and comparison.

In [4]:
with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\nSaved to evaluation_results.json")


Saved to evaluation_results.json


##  Analysis

The model shows acceptable performance, but there are some limitations:

- The **precision is relatively high**, meaning most detected cars are correct.
- The **recall is lower**, indicating that the model misses some cars.

### Why did this happen?

Although YOLO is pretrained on the "car" class, the model performance depends heavily on the training data.

In this case:
- The dataset was **biased toward top-view vehicles**
- The model learned specific patterns instead of general car features
- As a result, it struggled with detecting **side-view cars**

### Key Insight

> The model is only as good as the data it is trained on.

### Conclusion

To improve performance, a new model should be trained using more diverse data, including different vehicle angles (top-view and side-view), to improve generalization and increase recall.